In [ ]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/FRAMES'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Saving to: {os.path.abspath(OUTPUT_DIR)}')

Mounted at /content/drive
Saving to: /content/drive/MyDrive/FRAMES


In [ ]:
!pip install rank_bm25 datasets wikipedia tqdm -q

  Preparing metadata (setup.py) ... done


In [ ]:
import json
import pickle
import time
import os
import re
import ast
import collections
from tqdm.auto import tqdm
import wikipedia
from datasets import load_dataset
from rank_bm25 import BM25Okapi

wikipedia.set_lang('en')
def get_wiki_links(row):
    links = row['wiki_links']
    if isinstance(links, str):
        try:
            links = ast.literal_eval(links)
        except:
            links = [links]
    return [l for l in links if isinstance(l, str) and len(l) > 3]

print('Imports OK')

Imports OK


In [ ]:
# Load FRAMES
ds = load_dataset('google/frames-benchmark')
frames = ds['test']
print(f'Loaded {len(frames)} questions')
print('\nColumns:', frames.column_names)
print('\nExample:')
print('  Question:', frames[0]['Prompt'][:100])
print('  Answer:', frames[0]['Answer'])
print('  Wiki links:', frames[0]['wiki_links'][:2])
print('  Reasoning types:', frames[0]['reasoning_types'])

Loading FRAMES dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

test.tsv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/824 [00:00<?, ? examples/s]

Loaded 824 questions

Columns: ['Unnamed: 0', 'Prompt', 'Answer', 'wikipedia_link_1', 'wikipedia_link_2', 'wikipedia_link_3', 'wikipedia_link_4', 'wikipedia_link_5', 'wikipedia_link_6', 'wikipedia_link_7', 'wikipedia_link_8', 'wikipedia_link_9', 'wikipedia_link_10', 'wikipedia_link_11+', 'reasoning_types', 'wiki_links']

Example:
  Question: If my future wife has the same first name as the 15th first lady of the United States' mother and he
  Answer: Jane Ballou
  Wiki links: ['
  Reasoning types: Multiple constraints


In [ ]:
def title_from_url(url):
    """Gets titles from the URL, forexample https://en.wikipedia.org/wiki/Albert_Einstein -> Albert Einstein"""
    match = re.search(r'/wiki/(.+)', url)
    if match:
        return match.group(1).replace('_', ' ')
    return url

# Collect all unique gold article titles
gold_titles = set()
for row in frames:
    for url in get_wiki_links(row):
        gold_titles.add(title_from_url(url))

print(f'Unique gold article titles: {len(gold_titles)}')
print('Sample:', list(gold_titles)[:5])

Unique gold article titles: 2507
Sample: ['Grease 2', 'Alastair Galbraith', 'Courteney Cox', 'Taylor Swift albums discography', 'Pernicious anemia']


In [ ]:
def fetch_article(title):
    try:
        page = wikipedia.page(title, auto_suggest=False)
        return page.title, page.content
    except wikipedia.exceptions.DisambiguationError as e:
        try:
            page = wikipedia.page(e.options[0], auto_suggest=False)
            return page.title, page.content
        except:
            return None
    except:
        return None

corpus = {}
failed = []

print(f'Fetching {len(gold_titles)} gold articles')
for title in tqdm(gold_titles):
    result = fetch_article(title)
    if result:
        corpus[result[0]] = result[1]
    else:
        failed.append(title)
    time.sleep(0.1)

print(f'\nFetched: {len(corpus)} gold articles')
print(f'Failed: {len(failed)}')
if failed:
    print('Failed titles (first 10):', failed[:10])

# Implementation to retry failed articles
if failed:
    print(f'Retrying {len(failed)} failed articles')
    import urllib.parse
    still_failed = []
    for title in tqdm(failed):
        time.sleep(0.5)
        result = fetch_article(title)
        if result is None:
            decoded = urllib.parse.unquote(title)
            if decoded != title:
                result = fetch_article(decoded)
        if result:
            corpus[result[0]] = result[1]
        else:
            still_failed.append(title)
    recovered = len(failed) - len(still_failed)
    print(f'Retry recovered {recovered} articles. Permanently failed: {len(still_failed)}')
    if still_failed[:10]:
        print('Permanently failed (first 10):', still_failed[:10])
    failed = still_failed

Fetching 2507 gold articles (this takes a few minutes)...


  0%|          | 0/2507 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')



Fetched: 2311 gold articles
Failed: 168
Failed titles (first 10): ['Wroc%C5%82aw', 'List of S%26P 500 companies', 'Bront%C3%AB family', 'Schindler%27s List', 'List of minor planets: 8001%E2%80%939000#323c', '2019%E2%80%9320 NBA season', 'King%27s Service Medal', '2016 CONCACAF Women%27s Olympic Qualifying Championship#Group A', '1984%E2%80%9385 NHL season', 'Jo%C3%ABl Retornaz']
Retrying 168 failed articles (longer delay + URL-decode fallback)...


  0%|          | 0/168 [00:00<?, ?it/s]

Retry recovered 154 articles. Permanently failed: 14
Permanently failed (first 10): ['Nemanja Markovi%C4%87', 'Jack Vance (tennis)', 'Transformers: Dark of the Moon, https://en.wikipedia.org/wiki/Pain %26 Gain, https://en.wikipedia.org/wiki/Transformers: Age of Extinction, https://en.wikipedia.org/wiki/13 Hours: The Secret Soldiers of Benghazi, https://en.wikipedia.org/wiki/Transformers: The Last Knight, https://en.wikipedia.org/wiki/6 Underground (film), https://en.wikipedia.org/wiki/Ambulance (2022 film)', 'Pok%C3%A9mon (NOT REQUIRED, BUT HELPFUL) ', 'Summer Magic (film), https://en.wikipedia.org/wiki/The Incredible Journey (film), https://en.wikipedia.org/wiki/The Sword in the Stone (1963 film)', 'https://en.wikipedia.org/w/index.php?title=Bronco&redirect=no', '2021 French Open %E2%80%93 Men%2527s singles', 'Tim Salmon, https://en.wikipedia.org/wiki/Troy Glaus', 'American Family Field, https://en.wikipedia.org/wiki/LoanDepot Park, https://en.wikipedia.org/wiki/Globe Life Field, ', '

In [ ]:
# Add random Wikipedia articles as distractors
# These are unrelated articles that simulate a real-world corpus
NUM_DISTRACTORS = 1500
distractor_count = 0
print(f'Fetching {NUM_DISTRACTORS} random distractor articles...')
pbar = tqdm(total=NUM_DISTRACTORS)
while distractor_count < NUM_DISTRACTORS:
    try:
        rand_title = wikipedia.random(pages=1)
        if rand_title not in corpus:
            result = fetch_article(rand_title)
            if result:
                corpus[result[0]] = result[1]
                distractor_count += 1
                pbar.update(1)
        time.sleep(0.05)
    except:
        pass
pbar.close()

print(f'\nTotal corpus size: {len(corpus)} articles')
print(f'  Gold: ~{len(gold_titles)}')
print(f'  Distractors: ~{NUM_DISTRACTORS}')

Fetching 1500 random distractor articles...


  0%|          | 0/1500 [00:00<?, ?it/s]


Total corpus size: 3963 articles
  Gold: ~2507
  Distractors: ~1500


In [ ]:
# Build BM25 index
print('Building BM25 index...')
titles = list(corpus.keys())
tokenized_corpus = [corpus[t].lower().split() for t in tqdm(titles)]
bm25 = BM25Okapi(tokenized_corpus)
print(f'BM25 index built over {len(titles)} documents')

Building BM25 index...


  0%|          | 0/3963 [00:00<?, ?it/s]

BM25 index built over 3963 documents


In [ ]:
def retrieve_bm25(query, k=5):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = scores.argsort()[-k:][::-1]
    return [(titles[i], corpus[titles[i]]) for i in top_idx]

def recall_at_k(question, gold_urls, k=5):
    gold = set(title_from_url(url).lower() for url in gold_urls)
    if not gold:
        return 1.0
    retrieved = set(t.lower() for t, _ in retrieve_bm25(question, k=k))
    return len(retrieved & gold) / len(gold)

example = frames[0]
print('Question:', example['Prompt'][:80])
print('Gold articles:', [title_from_url(u) for u in get_wiki_links(example)])
print('Top-5 BM25:', [t for t, _ in retrieve_bm25(example['Prompt'], k=5)])

Question: If my future wife has the same first name as the 15th first lady of the United S
Gold articles: ['President of the United States', 'James Buchanan', 'Harriet Lane', 'List of presidents of the United States who died in office', 'James A. Garfield']
Top-5 BM25: ['Pride and Prejudice', 'Charles III', 'Edith Wilson', 'List of presidents of the United States who died in office', 'Prince Philip, Duke of Edinburgh']


In [ ]:
# Checking recall for different k-values
results = {k: [] for k in [1, 3, 5, 10]}

print('Evaluating Recall@k on all FRAMES questions')
for row in tqdm(frames):
    links = get_wiki_links(row)
    for k in [1, 3, 5, 10]:
        r = recall_at_k(row['Prompt'], links, k=k)
        results[k].append(r)

print('\n BM25 Recall@k:')
for k in [1, 3, 5, 10]:
    avg = sum(results[k]) / len(results[k])
    print(f'  Recall@{k:2d}: {avg:.3f}')

Evaluating Recall@k on all FRAMES questions...


  0%|          | 0/824 [00:00<?, ?it/s]


--- BM25 Recall@k ---
  Recall@ 1: 0.212
  Recall@ 3: 0.393
  Recall@ 5: 0.449
  Recall@10: 0.508

Expected from FRAMES paper: ~0.12-0.15 per retrieval step
(Low recall is expected — this is why multi-step retrieval helps)


In [ ]:
# #Recall based on number of gold articles
by_n_articles = collections.defaultdict(list)
for row in frames:
    links = get_wiki_links(row)
    n = len(links)
    r = recall_at_k(row['Prompt'], links, k=5)
    by_n_articles[n].append(r)

print('Recall@5 by number of required articles:')
for n in sorted(by_n_articles.keys()):
    vals = by_n_articles[n]
    print(f'  {n} articles: {sum(vals)/len(vals):.3f}  (n={len(vals)} questions)')

Recall@5 by number of required articles:
  2 articles: 0.527  (n=310 questions)
  3 articles: 0.453  (n=288 questions)
  4 articles: 0.354  (n=134 questions)
  5 articles: 0.378  (n=37 questions)
  6 articles: 0.254  (n=19 questions)
  7 articles: 0.297  (n=13 questions)
  8 articles: 0.438  (n=6 questions)
  9 articles: 0.259  (n=3 questions)
  10 articles: 0.167  (n=3 questions)
  11 articles: 0.198  (n=11 questions)


In [ ]:
# Save everything to disk
corpus_path = os.path.join(OUTPUT_DIR, 'corpus.json')
titles_path = os.path.join(OUTPUT_DIR, 'titles.json')
bm25_path   = os.path.join(OUTPUT_DIR, 'bm25_index.pkl')

with open(corpus_path, 'w', encoding='utf-8') as f:
    json.dump(corpus, f, ensure_ascii=False)

with open(titles_path, 'w', encoding='utf-8') as f:
    json.dump(titles, f, ensure_ascii=False)

with open(bm25_path, 'wb') as f:
    pickle.dump(bm25, f)

print('Saved:')
print(f'  {corpus_path}  ({os.path.getsize(corpus_path)//1024} KB)')
print(f'  {titles_path}')
print(f'  {bm25_path}')

Saved:
  /content/drive/MyDrive/FRAMES/corpus.json  (62304 KB)
  /content/drive/MyDrive/FRAMES/titles.json
  /content/drive/MyDrive/FRAMES/bm25_index.pkl

Done. Run notebook 02 next.
